In [1]:
import os
import pandas as pd
import numpy as np
from y0.graph import NxMixedGraph

import pyro
from causomic.simulation.example_graphs import frontdoor
from causomic.simulation.proteomics_simulator import simulate_data
from causomic.causal_model.LVM import LVM

/root/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
graph = frontdoor()

In [3]:
data = simulate_data(
    graph["Networkx"],
    graph["Coefficients"],
    include_missing= False,
    cell_type = False,
    n_cells = 3,
    mar_missing_param = 0.05,
    mnar_missing_param = [-3, 0.4],
    add_error = False,
    error_node = None,
    intervention = {},
    add_feature_var = True,
    n = 500,
    seed= 1,
) 



simulating data...
adding feature level data...


/mnt/e/OneDrive - Northeastern University/Northeastern/Research/Causal_Inference/MS_causal_inference/Causomic/src/causomic/simulation/proteomics_simulator.py:579: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  feature_level_data = pd.concat(
/mnt/e/OneDrive - Northeastern University/Northeastern/Research/Causal_Inference/MS_causal_inference/Causomic/src/causomic/simulation/proteomics_simulator.py:579: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  feature_level_data = pd.concat(
/mnt/e/OneDrive - 

masking data...


In [4]:
input_data = data["Protein_data"]

In [5]:
import networkx as nx
obs_nodes = graph["Networkx"].nodes
    
attrs = {node: (True if node not in obs_nodes and 
                node != "\\n" else False) for node in obs_nodes}

nx.set_node_attributes(graph["Networkx"], attrs, name="hidden")
# Use y0 to build ADMG
dag = NxMixedGraph()
dag = dag.from_latent_variable_dag(graph["Networkx"], "hidden")

In [7]:
# Appears to be computationally expensive
pyro.clear_param_store()

# Full imp Bayesian model
lvm = LVM(backend="pyro",
        num_steps=2000,
        patience=30,
        initial_lr=1.5,
        verbose=True)

lvm.fit(pd.DataFrame(input_data), dag)

Fitting LVM with pyro backend
Data shape: (500, 4)
Parsing causal graph structure...
Scaling observational data...
Found 1 root nodes and 3 descendant nodes
Processing observational data...
Setting up prior distributions...
Starting SVI training for 2000 steps with early stopping
Patience: 30, Min delta: 5
Step 0: loss=145913.1483  ema=145913.1483
Step 100: loss=414.0441  ema=2227.9568
Step 200: loss=-1607.2664  ema=-1288.0065
Step 300: loss=-3266.4426  ema=-3039.5690
Step 400: loss=-4386.3223  ema=-4221.5840
Step 500: loss=-5102.3348  ema=-4991.4249
Step 600: loss=-5722.3654  ema=-5601.7678
Step 700: loss=-6304.4637  ema=-6167.0969
Step 800: loss=-6793.7677  ema=-6679.5493
Step 900: loss=-7273.1861  ema=-7164.7632
Step 1000: loss=-7712.8309  ema=-7616.2699
Step 1100: loss=-8126.7602  ema=-8049.2325
Step 1200: loss=-8525.2913  ema=-8448.3378
Step 1300: loss=-8957.5087  ema=-8859.1807
Step 1400: loss=-9339.4848  ema=-9282.8163
Step 1500: loss=-9751.8265  ema=-9672.1205
Step 1600: loss=-

In [85]:
lvm.intervention({"X": 1}, ["Z"])

In [86]:
lvm.intervention_samples.mean() - lvm.posterior_samples.mean()

Z    0.692536
dtype: float64

In [59]:
graph["Coefficients"]

{'C': {'intercept': 6, 'error': 1.0},
 'X': {'intercept': 1, 'error': 1.0, 'C': 0.5},
 'Y': {'intercept': 1.6, 'error': 0.25, 'X': 0.5},
 'Z': {'intercept': -3, 'error': 0.25, 'Y': 1.0, 'C': 0.5}}